# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [67]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [68]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 106.6 KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: <StringArray>
['Asia', 'Europe', 'Africa', 'Americas', 'Oceania']
Length: 5, dtype: str
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [69]:
import plotly.express as px
import pandas as pd

# Load dataset
df = px.data.gapminder()

# Filter for Europe, 2002 & 2007
df_subset = df[
    (df["continent"] == "Europe") &
    (df["year"].isin([2002, 2007]))
].copy()

# Convert year to string for discrete colors
df_subset["year"] = df_subset["year"].astype(str)

# ✅ SCATTER ONLY (no size parameter)
fig = px.scatter(
    df_subset,
    x="gdpPercap",
    y="lifeExp",
    color="year",
    hover_name="country",
    log_x=True,
    opacity=0.8,
    color_discrete_map={
        "2002": "#4C72B0",  # blue
        "2007": "#DD8452"   # orange
    },
    title="Europe’s Progress: GDP vs Life Expectancy (2002–2007)"
)

# Make markers consistent size (true scatter)
fig.update_traces(marker=dict(size=10))

# Layout polish
fig.update_layout(
    title=dict(x=0.5, font=dict(size=20)),
    xaxis_title="GDP per Capita (log scale)",
    yaxis_title="Life Expectancy",
    legend_title="Year",
    template="plotly_white"
)

# ✅ Clear annotation (visible area)
fig.add_annotation(
    x=25000,
    y=80,
    text="Shift upward and right →<br>economic growth & better health",
    showarrow=True,
    arrowhead=2,
    ax=-120,
    ay=-60,
    font=dict(size=12),
    bgcolor="white",
    bordercolor="black"
)

fig.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [70]:
import plotly.express as px
import pandas as pd
import math

# Load data
df = px.data.gapminder()

# Filter 2007 data
df_2007 = df[df["year"] == 2007].copy()

# Highlight continent (choose ONE)
highlight = "Asia"

# Create grouping
df_2007["group"] = df_2007["continent"].apply(
    lambda x: highlight if x == highlight else "Other"
)

# Create plot
fig = px.scatter(
    df_2007,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    hover_name="country",
    color="group",
    log_x=True,
    size_max=50,
    opacity=0.6,
    color_discrete_map={
        highlight: "#E45756",   # strong highlight
        "Other": "#C7C7C7"      # soft grey
    },
    title="Asia’s Growth Story: High Population, Rising Life Expectancy (2007)",
    custom_data=['country', 'pop']
)

# Layout polish
fig.update_layout(
    showlegend=False,
    title=dict(x=0.5, font=dict(size=20)),
    xaxis_title="GDP per Capita (USD in log scale)",
    yaxis_title="Life Expectancy",
    template="plotly_white"
)

fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>'
                  'GDP per capita: $%{x:,.0f}<br>'
                  'Life expectancy: %{y:.1f} yrs<br>'
                  'Population: %{customdata[1]:,.0f}')

# 🔥 Strong, visible annotation (PLACED IN EMPTY SPACE)
fig.add_annotation(
    x=math.log10(3000), y=47,
    text="<b>Asia stands out</b><br>Large populations span a wide GDP range,<br>yet life expectancy continues to rise",
    showarrow=False,
    font=dict(color='#E63946', size=11, family='Arial'),
    bgcolor='white', bordercolor='#E63946', borderwidth=1, borderpad=4
)


fig.show()